In [1]:
import pyarrow.parquet as pq
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
data = pq.ParquetFile("C:\\Users\\20244416\\OneDrive - TU Eindhoven\\Desktop\\Multidisciplinary CBL\\data\\uk_crime_full_cleaned.parquet")

print(data.schema)

required group field_id=-1 duckdb_schema {
  optional binary field_id=-1 Crime ID (String);
  optional binary field_id=-1 Month (String);
  optional binary field_id=-1 Reported by (String);
  optional binary field_id=-1 Falls within (String);
  optional double field_id=-1 Longitude;
  optional double field_id=-1 Latitude;
  optional binary field_id=-1 Location (String);
  optional binary field_id=-1 LSOA code (String);
  optional binary field_id=-1 LSOA name (String);
  optional binary field_id=-1 Crime type (String);
  optional binary field_id=-1 Last outcome category (String);
  optional boolean field_id=-1 COVID period;
}



In [3]:
columnsNeeded = ['LSOA name', 'Month', 'Crime type']

df = pd.read_parquet("C:\\Users\\20244416\\OneDrive - TU Eindhoven\\Desktop\\Multidisciplinary CBL\\data\\uk_crime_full_cleaned.parquet", columns=columnsNeeded)

In [4]:
df

,LSOA name,Month,Crime type
0,Bath and North East Somerset 001A,2017-07,Vehicle crime
1,Bath and North East Somerset 004A,2017-07,Anti-social behaviour
2,Bath and North East Somerset 007A,2017-07,Anti-social behaviour
3,Bath and North East Somerset 007B,2017-07,Violence and sexual offences
4,Bath and North East Somerset 007F,2017-07,Anti-social behaviour
...,...,...,...
50171394,Wiltshire 060E,2026-01,Criminal damage and arson
50171395,Wiltshire 061A,2026-01,Violence and sexual offences
50171396,Wiltshire 062B,2026-01,Criminal damage and arson
50171397,Wiltshire 065B,2026-01,Anti-social behaviour


---

In [5]:
def get_crime_count(crime_type, area, timeColumn: str) -> list:
    '''
    Given an area and a crime type, the function returns a list of crime counts for each month given.
    '''
    filteredData = df[(df['LSOA name'] == area) & (df['Crime type'] == crime_type)]
    all_months = sorted(df[timeColumn].unique())
    crimeCount = filteredData[timeColumn].value_counts()
    crimeCount = crimeCount.reindex(all_months, fill_value=0)
    
    return list(crimeCount)

# get_crime_count('Bicycle theft', 'City of London 001F', 'Month')

In [6]:
#Sort the data
df = df.sort_values(by=['LSOA name', 'Month'])

In [7]:
violenceCount = get_crime_count('Violence', area, 'Month')

NameError: name 'area' is not defined

In [ ]:
# totalCrimeCount =

In [ ]:
# brokerageScore = 

In [ ]:

#Create the "Future Violence" column (The T+1 Lag)
# We group by LSOA so we don't accidentally shift one neighborhood's data into another
df['violenceNextMonth'] = df.groupby('LSOA name')['violenceCount'].shift(-1)

df = df.dropna(subset=['violenceNextMonth'])

In [ ]:
equation = "violenceNextMonth ~ brokerageScore + totalCrimeCount"

model = smf.glm(formula=equation, 
                data=df, 
                family=smf.families.NegativeBinomial(alpha=1.0)).fit()

print(model.summary())